# EDA: Two-Period Cascading Flow Analysis (Adding 2011OD data)
**Strategy: Fixed 2010 IMD Baseline × 2011 & 2021 Census O-D Comparison**

### Methodology
- **Cascading Flow Index (two periods)**: Uses **IMD 2010** (fixed baseline) to assign
  wealth deciles to MSOAs, then measures cascade flows from **both** the **2011** and
  **2021 Census O-D** data under the same classification.
- **Primary gentrification signal**: ΔCFI = CFI₂₀₂₁ − CFI₂₀₁₁ per MSOA. Positive ΔCFI
  indicates intensifying cascading displacement over the decade.
- **MSOA typology**: Classify MSOAs by (CFI₂₀₁₁, CFI₂₀₂₁) into emerging, sustained,
  stalled, and stable gentrification categories.
- **Temporal Validation**: Independently compute IMD change (2010 → 2019) per MSOA.
  Test whether ΔCFI correlates with deprivation decline (three-way triangulation).
- **Robustness check**: Re-compute 2021 flows using IMD 2019 deciles to confirm
  results are not artefacts of baseline choice.
- **Geography**: All analysis harmonised to **2011 MSOA codes**. 2011 O-D data uses
  native 2011 codes (aggregated from OA); 2021 O-D data mapped via ONS
  MSOA 2011-to-2021 lookup. Split/merged MSOAs excluded.

---

### Structure
- **Part A (Sections 1–6)**: Data loading, preprocessing & aggregation — no plots
- **Part B (Sections 7–14)**: Analysis, visualisation & export

---

### Data Sources
| File | Description |
|------|-------------|
| `imd_2010.xls` | IMD 2010 at LSOA level |
| `imd_2019.csv` | IMD 2019 at LSOA level |
| `ODMG01EW_MSOA.csv` | 2021 Census migration O-D (MSOA level) |
| `census_od_2011_oa.csv` | 2011 Census migration O-D (OA level, from WICID) |
| `NSPCL_NOV22_UK_LU.csv` | Postcode lookup (2011 geographies) |
| `msoa_2011_to_2021_lookup.csv` | MSOA 2011 ↔ 2021 code mapping |

---
# PART A — DATA LOADING & PREPROCESSING
---

## 1. Setup & Load All Datasets

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pyprojroot import here
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.1)

ROOT = here()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ---- File paths ----
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
imd_2019_path       = DATA_DIR / 'imd_2019.csv'
census_od_2021_path = DATA_DIR / 'ODMG01EW_MSOA.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'
census_od_2011_path = DATA_DIR / 'mf01ew_oa_all_v1.csv'  

# ---- Column definitions (from MF01EW metadata) ----
# Col A = "Area of usual residence"      = DESTINATION
# Col B = "Area of address 1 year ago"   = ORIGIN
# Col C = "Persons"                       = COUNT
OD_2011_ORIGIN_COL = 'origin_oa'
OD_2011_DEST_COL   = 'dest_oa'
OD_2011_COUNT_COL  = 'persons'

# ---- Load (no header in raw file) ----
census_od_2011_raw = pd.read_csv(
    census_od_2011_path,
    header=None,
    names=['dest_oa', 'origin_oa', 'persons'],  # A=dest, B=origin, C=count
    dtype={'dest_oa': str, 'origin_oa': str, 'persons': int}
)

# Log special codes before they drop out
special_mask = census_od_2011_raw['origin_oa'].str.startswith('OD')
print(f'Rows with special origin codes (cross-border etc): '
      f'{special_mask.sum():,} — these will be excluded')

In [ ]:
# ---- Load IMD ----
imd_2019 = pd.read_csv(imd_2019_path)
imd_2019.columns = imd_2019.columns.str.strip()

imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()

# ---- Load Census O-D 2021 (MSOA level) ----
census_od_2021 = pd.read_csv(census_od_2021_path)

# ---- Load Census O-D 2011 (OA level) ----
census_od_2011_raw = pd.read_csv(census_od_2011_path)

# ---- Load Lookups ----
lookup = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
msoa_11_21 = pd.read_csv(msoa_lookup_path)

# Quick overview
for name, df in [('IMD 2010', imd_2010), ('IMD 2019', imd_2019),
                  ('Census O-D 2021', census_od_2021),
                  ('Census O-D 2011 (OA)', census_od_2011_raw),
                  ('MSOA Lookup', msoa_11_21)]:
    print(f'\n===== {name} =====')
    print(f'Shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    display(df.head(2))

---
## 2. Geography Harmonisation

Two separate pipelines:

**2021 O-D**: MSOA-level data uses 2021 codes → map to 2011 MSOA codes via the
ONS MSOA 2011-to-2021 lookup. Only unchanged (1:1) MSOAs are kept.

**2011 O-D**: OA-level data uses native 2011 codes → aggregate to 2011 MSOA codes
using the OA-to-MSOA mapping from the postcode lookup. No cross-year harmonisation needed.

In [ ]:
# ---- 2a. MSOA 2021 → 2011 mapping (for 2021 O-D data) ----
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()

unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']].copy()
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))

print(f'Total MSOAs in lookup: {len(msoa_11_21)}')
print(f'Unchanged (usable):    {len(unchanged)}')
print(f'Dropped (split/merged): {len(msoa_11_21) - len(unchanged)}')

In [ ]:
# ---- 2b. OA → MSOA mapping (for 2011 O-D data) ----
# Extract unique OA-to-MSOA assignments from the postcode lookup
oa_to_msoa = (
    lookup[['oa11cd', 'msoa11cd']]
    .drop_duplicates()
    .dropna()
)
# Sanity check: each OA should map to exactly one MSOA
oa_dup = oa_to_msoa.groupby('oa11cd')['msoa11cd'].nunique()
assert (oa_dup == 1).all(), f'{(oa_dup > 1).sum()} OAs map to multiple MSOAs!'
oa_to_msoa_dict = dict(zip(oa_to_msoa['oa11cd'], oa_to_msoa['msoa11cd']))

print(f'OA-to-MSOA mappings: {len(oa_to_msoa_dict):,}')

In [ ]:
# ---- 2c. Map 2021 O-D to 2011 MSOA codes ----
ORIGIN_COL_2021 = 'Migrant MSOA one year ago code'
DEST_COL_2021   = 'Middle layer Super Output Areas code'

census_od_2021['origin_msoa11'] = census_od_2021[ORIGIN_COL_2021].map(msoa21_to_11)
census_od_2021['dest_msoa11']   = census_od_2021[DEST_COL_2021].map(msoa21_to_11)

n_total = len(census_od_2021)
n_mapped = census_od_2021[['origin_msoa11', 'dest_msoa11']].notna().all(axis=1).sum()
print(f'2021 O-D records: {n_total:,}')
print(f'Both endpoints mapped to 2011 codes: {n_mapped:,} ({n_mapped/n_total*100:.1f}%)')

In [ ]:
# ---- 2d. Aggregate 2011 O-D from OA to MSOA level ----
census_od_2011 = census_od_2011_raw.copy()

# Map OA codes to MSOA codes
census_od_2011['origin_msoa11'] = census_od_2011[OD_2011_ORIGIN_COL].map(oa_to_msoa_dict)
census_od_2011['dest_msoa11']   = census_od_2011[OD_2011_DEST_COL].map(oa_to_msoa_dict)

n_total_11 = len(census_od_2011)
n_mapped_11 = census_od_2011[['origin_msoa11', 'dest_msoa11']].notna().all(axis=1).sum()
print(f'2011 O-D records (OA level): {n_total_11:,}')
print(f'Both endpoints mapped to MSOA: {n_mapped_11:,} ({n_mapped_11/n_total_11*100:.1f}%)')

# Aggregate to MSOA-to-MSOA flows
census_od_2011_msoa = (
    census_od_2011
    .dropna(subset=['origin_msoa11', 'dest_msoa11'])
    .groupby(['origin_msoa11', 'dest_msoa11'])[OD_2011_COUNT_COL]
    .sum()
    .reset_index()
    .rename(columns={OD_2011_COUNT_COL: 'count'})
)

# Remove non-movers (same origin and destination MSOA)
census_od_2011_msoa = census_od_2011_msoa[
    census_od_2011_msoa['origin_msoa11'] != census_od_2011_msoa['dest_msoa11']
].copy()

print(f'Aggregated MSOA-to-MSOA flow records (2011): {len(census_od_2011_msoa):,}')
print(f'Total migrants (2011): {census_od_2011_msoa["count"].sum():,.0f}')

---
## 3. London Filter & IMD Aggregation to MSOA

In [ ]:
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

london_lookup = (
    lookup[lookup['ladnm'].isin(london_boroughs)]
    [['lsoa11cd', 'msoa11cd', 'ladnm']]
    .drop_duplicates()
)

# Set of London MSOA codes (used for filtering flows)
london_msoas = set(london_lookup['msoa11cd'].unique())

print(f'London LSOAs: {london_lookup["lsoa11cd"].nunique()}')
print(f'London MSOAs: {len(london_msoas)}')
print(f'Boroughs:     {london_lookup["ladnm"].nunique()}')

In [ ]:
# ---- IMD 2010 → MSOA ----
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

imd_2010_london = pd.merge(
    imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]],
    london_lookup,
    left_on=IMD_2010_LSOA_COL, right_on='lsoa11cd'
)
msoa_imd_2010 = (
    imd_2010_london
    .groupby(['msoa11cd', 'ladnm'])[IMD_2010_SCORE_COL]
    .mean().reset_index()
    .rename(columns={IMD_2010_SCORE_COL: 'IMD_2010'})
)

# ---- IMD 2019 → MSOA ----
imd_2019_london = pd.merge(
    imd_2019[['LSOA code (2011)', 'Index of Multiple Deprivation (IMD) Score']],
    london_lookup,
    left_on='LSOA code (2011)', right_on='lsoa11cd'
)
msoa_imd_2019 = (
    imd_2019_london
    .groupby('msoa11cd')['Index of Multiple Deprivation (IMD) Score']
    .mean().reset_index()
    .rename(columns={'Index of Multiple Deprivation (IMD) Score': 'IMD_2019'})
)

# ---- Combine ----
msoa_wealth = pd.merge(msoa_imd_2010, msoa_imd_2019, on='msoa11cd', how='inner')
msoa_wealth['IMD_Change'] = msoa_wealth['IMD_2019'] - msoa_wealth['IMD_2010']

print(f'London MSOAs with both IMD years: {len(msoa_wealth)}')
msoa_wealth.head()

---
## 4. Wealth Decile Assignment (Fixed 2010 Baseline)

Deciles are assigned using **IMD 2010 only**. This fixed classification is applied
identically to **both** the 2011 and 2021 O-D data, so any difference in cascade
flows is attributable to changes in migration patterns, not area reclassification.

Convention: 1 = most deprived, 10 = least deprived (wealthiest).

In [ ]:
msoa_wealth['Wealth_Decile'] = pd.qcut(
    msoa_wealth['IMD_2010'], 10, labels=False
) + 1
msoa_wealth['Wealth_Decile'] = 11 - msoa_wealth['Wealth_Decile']

wealth_dict = msoa_wealth.set_index('msoa11cd')['Wealth_Decile'].to_dict()

decile_counts = msoa_wealth['Wealth_Decile'].value_counts().sort_index()
print(f'Average MSOAs per decile: {decile_counts.mean():.1f}')
print(decile_counts)

In [ ]:
# ---- Robustness: IMD 2019 deciles (for later sensitivity check) ----
msoa_wealth['Wealth_Decile_2019'] = pd.qcut(
    msoa_wealth['IMD_2019'], 10, labels=False
) + 1
msoa_wealth['Wealth_Decile_2019'] = 11 - msoa_wealth['Wealth_Decile_2019']

wealth_dict_2019 = msoa_wealth.set_index('msoa11cd')['Wealth_Decile_2019'].to_dict()

print('IMD 2019 deciles computed (for robustness check in Part B).')

---
## 5. O-D Flow Processing: Build London Flow Tables for Both Years

A reusable function processes either year's O-D data into:
- London-to-London flow table with decile assignments
- Flow matrix (decile × decile)
- Flow direction summary
- Per-MSOA cascade features

In [ ]:
def build_london_flows(od_df, origin_col, dest_col, count_col, wealth_mapping,
                        london_msoa_set, year_label):
    """
    Process O-D data into London-to-London flows with decile assignments.
    
    Parameters
    ----------
    od_df : DataFrame with origin_msoa11, dest_msoa11 already mapped
            (or raw MSOA codes in origin_col/dest_col)
    origin_col, dest_col : column names for origin/dest MSOA 2011 codes
    count_col : column name for migrant count
    wealth_mapping : dict mapping msoa11cd → Wealth_Decile
    london_msoa_set : set of London MSOA codes
    year_label : str, e.g. '2011' or '2021'
    
    Returns
    -------
    london_flow : DataFrame of London-to-London flows with decile info
    flow_matrix : pivot table (origin_decile × dest_decile)
    flow_direction : Series with upward/downward/lateral/total counts
    shift_dist : Series indexed by Decile_Shift
    """
    df = od_df.copy()
    
    # Assign deciles
    df['Origin_Decile'] = df[origin_col].map(wealth_mapping)
    df['Dest_Decile']   = df[dest_col].map(wealth_mapping)
    
    # Keep only London-to-London flows with valid deciles
    london_flow = df.dropna(subset=['Origin_Decile', 'Dest_Decile']).copy()
    london_flow['Origin_Decile'] = london_flow['Origin_Decile'].astype(int)
    london_flow['Dest_Decile']   = london_flow['Dest_Decile'].astype(int)
    london_flow['Decile_Shift']  = london_flow['Dest_Decile'] - london_flow['Origin_Decile']
    
    # Flow matrix
    flow_matrix = london_flow.pivot_table(
        index='Origin_Decile', columns='Dest_Decile',
        values=count_col, aggfunc='sum', fill_value=0
    )
    
    # Direction summary
    total = london_flow[count_col].sum()
    upward   = london_flow[london_flow['Decile_Shift'] > 0][count_col].sum()
    downward = london_flow[london_flow['Decile_Shift'] < 0][count_col].sum()
    lateral  = london_flow[london_flow['Decile_Shift'] == 0][count_col].sum()
    
    flow_direction = pd.Series({
        'Upward': upward, 'Downward': downward,
        'Lateral': lateral, 'Total': total
    })
    
    # Shift distribution
    shift_dist = london_flow.groupby('Decile_Shift')[count_col].sum()
    
    print(f'\n=== {year_label} Flow Summary ===')
    print(f'  London-to-London records: {len(london_flow):,}')
    print(f'  Total migrants:           {total:,.0f}')
    print(f'  Upward:  {upward:>10,.0f} ({upward/total*100:.1f}%)')
    print(f'  Down:    {downward:>10,.0f} ({downward/total*100:.1f}%)')
    print(f'  Lateral: {lateral:>10,.0f} ({lateral/total*100:.1f}%)')
    
    return london_flow, flow_matrix, flow_direction, shift_dist

In [ ]:
def compute_cascade_features(london_flow, count_col, origin_msoa_col, dest_msoa_col):
    """
    Compute per-MSOA cascade features from a London flow table.
    
    Returns a DataFrame indexed by msoa11cd with columns:
    Inflow_Wealthier, Outflow_Poorer, Total_Inflow, Total_Outflow,
    Net_Cascade, Cascade_Ratio, Pct_Inflow_Wealthier
    """
    # Inflow from wealthier areas
    inflow_w = (
        london_flow[london_flow['Origin_Decile'] > london_flow['Dest_Decile']]
        .groupby(dest_msoa_col)[count_col].sum()
        .rename('Inflow_Wealthier')
    )
    
    # Outflow to more deprived areas
    outflow_p = (
        london_flow[london_flow['Dest_Decile'] < london_flow['Origin_Decile']]
        .groupby(origin_msoa_col)[count_col].sum()
        .rename('Outflow_Poorer')
    )
    
    # Total inflow & outflow
    total_in  = london_flow.groupby(dest_msoa_col)[count_col].sum().rename('Total_Inflow')
    total_out = london_flow.groupby(origin_msoa_col)[count_col].sum().rename('Total_Outflow')
    
    # Combine
    cascade = pd.DataFrame(index=inflow_w.index.union(outflow_p.index)
                                          .union(total_in.index)
                                          .union(total_out.index))
    cascade.index.name = 'msoa11cd'
    for s in [inflow_w, outflow_p, total_in, total_out]:
        cascade = cascade.join(s, how='left')
    cascade = cascade.fillna(0)
    
    # Derived features
    cascade['Net_Cascade'] = cascade['Inflow_Wealthier'] - cascade['Outflow_Poorer']
    cascade['Cascade_Ratio'] = np.where(
        cascade['Outflow_Poorer'] > 0,
        cascade['Inflow_Wealthier'] / cascade['Outflow_Poorer'],
        np.nan
    )
    cascade['Pct_Inflow_Wealthier'] = (
        cascade['Inflow_Wealthier'] / cascade['Total_Inflow'].replace(0, np.nan) * 100
    )
    
    return cascade

In [ ]:
# ---- 5a. Process 2021 O-D flows ----
od_2021 = census_od_2021.copy()
od_2021 = od_2021[od_2021[ORIGIN_COL_2021].astype(str) != '-8'].copy()

# Detect count column
count_cols_2021 = [c for c in od_2021.columns
                   if 'observation' in c.lower() or 'count' in c.lower()]
COUNT_COL_2021 = count_cols_2021[0] if count_cols_2021 else '_count'
if COUNT_COL_2021 == '_count':
    od_2021[COUNT_COL_2021] = 1

london_flow_2021, flow_matrix_2021, flow_dir_2021, shift_dist_2021 = build_london_flows(
    od_2021, 'origin_msoa11', 'dest_msoa11', COUNT_COL_2021,
    wealth_dict, london_msoas, '2021'
)

cascade_2021 = compute_cascade_features(
    london_flow_2021, COUNT_COL_2021, 'origin_msoa11', 'dest_msoa11'
)

In [ ]:
# ---- 5b. Process 2011 O-D flows ----
london_flow_2011, flow_matrix_2011, flow_dir_2011, shift_dist_2011 = build_london_flows(
    census_od_2011_msoa, 'origin_msoa11', 'dest_msoa11', 'count',
    wealth_dict, london_msoas, '2011'
)

cascade_2011 = compute_cascade_features(
    london_flow_2011, 'count', 'origin_msoa11', 'dest_msoa11'
)

---
## 6. Combine into Master Analysis Table & Compute ΔCFI

Merge 2011 and 2021 cascade features onto the MSOA wealth table,
compute the change in cascade pressure (ΔCFI), and assign typology labels.

In [ ]:
# ---- 6a. Merge cascade features from both years ----
# Suffix _11 and _21 for the two periods
msoa_analysis = msoa_wealth[['msoa11cd', 'ladnm', 'Wealth_Decile',
                              'Wealth_Decile_2019',
                              'IMD_2010', 'IMD_2019', 'IMD_Change']].copy()

# Rename cascade columns with year suffixes
c11 = cascade_2011.add_suffix('_11')
c21 = cascade_2021.add_suffix('_21')

msoa_analysis = (
    msoa_analysis
    .merge(c11, left_on='msoa11cd', right_index=True, how='left')
    .merge(c21, left_on='msoa11cd', right_index=True, how='left')
)
msoa_analysis = msoa_analysis.fillna(0)

print(f'MSOAs in analysis: {len(msoa_analysis)}')
print(f'  with 2011 flow data: {(msoa_analysis["Total_Inflow_11"] > 0).sum()}')
print(f'  with 2021 flow data: {(msoa_analysis["Total_Inflow_21"] > 0).sum()}')

In [ ]:
# ---- 6b. Compute ΔCFI (change in cascade pressure) ----
msoa_analysis['Delta_Net_Cascade'] = (
    msoa_analysis['Net_Cascade_21'] - msoa_analysis['Net_Cascade_11']
)
msoa_analysis['Delta_Pct_Inflow_Wealthier'] = (
    msoa_analysis['Pct_Inflow_Wealthier_21'] - msoa_analysis['Pct_Inflow_Wealthier_11']
)

print('ΔCFI computed (Net_Cascade and Pct_Inflow_Wealthier).')
msoa_analysis[['msoa11cd', 'Net_Cascade_11', 'Net_Cascade_21',
               'Delta_Net_Cascade']].describe().round(1)

In [ ]:
# ---- 6c. MSOA typology based on cascade pressure in both periods ----
# Threshold: Net_Cascade > 0 means net gentrification pressure
def classify_msoa(row):
    high_11 = row['Net_Cascade_11'] > 0
    high_21 = row['Net_Cascade_21'] > 0
    if not high_11 and high_21:
        return 'Emerging'
    elif high_11 and high_21:
        return 'Sustained'
    elif high_11 and not high_21:
        return 'Stalled'
    else:
        return 'Stable'

msoa_analysis['Gentrif_Type'] = msoa_analysis.apply(classify_msoa, axis=1)

print('=== MSOA Gentrification Typology ===')
print(msoa_analysis['Gentrif_Type'].value_counts())

In [ ]:
# ---- 6d. Decile-level summary (both years) ----
decile_summary = msoa_analysis.groupby('Wealth_Decile').agg(
    MSOA_Count=('msoa11cd', 'count'),
    Avg_Net_Cascade_11=('Net_Cascade_11', 'mean'),
    Avg_Net_Cascade_21=('Net_Cascade_21', 'mean'),
    Avg_Delta_CFI=('Delta_Net_Cascade', 'mean'),
    Avg_IMD_Change=('IMD_Change', 'mean'),
).round(1)

# ---- 6e. Borough-level summary ----
borough_summary = msoa_analysis.groupby('ladnm').agg(
    Num_MSOAs=('msoa11cd', 'count'),
    Avg_IMD_2010=('IMD_2010', 'mean'),
    Avg_IMD_Change=('IMD_Change', 'mean'),
    Total_Net_Cascade_11=('Net_Cascade_11', 'sum'),
    Total_Net_Cascade_21=('Net_Cascade_21', 'sum'),
    Total_Delta_CFI=('Delta_Net_Cascade', 'sum'),
    N_Emerging=('Gentrif_Type', lambda x: (x == 'Emerging').sum()),
    N_Sustained=('Gentrif_Type', lambda x: (x == 'Sustained').sum()),
).round(2).sort_values('Total_Delta_CFI', ascending=False)

# ---- 6f. Robustness: 2021 flows with IMD 2019 deciles ----
# Re-run 2021 flows with alternative decile assignment
_, flow_matrix_2021_alt, _, _ = build_london_flows(
    od_2021, 'origin_msoa11', 'dest_msoa11', COUNT_COL_2021,
    wealth_dict_2019, london_msoas, '2021 (IMD 2019 deciles)'
)
cascade_2021_alt = compute_cascade_features(
    _,  COUNT_COL_2021, 'origin_msoa11', 'dest_msoa11'
)

msoa_analysis = msoa_analysis.merge(
    cascade_2021_alt[['Net_Cascade']].rename(columns={'Net_Cascade': 'Net_Cascade_21_alt'}),
    left_on='msoa11cd', right_index=True, how='left'
).fillna(0)

print('\nPreprocessing complete. All tables ready for Part B.')
print(f'  msoa_analysis:  {msoa_analysis.shape}')
print(f'  decile_summary: {decile_summary.shape}')
print(f'  borough_summary:{borough_summary.shape}')

---
# PART B — ANALYSIS & VISUALISATION

All data tables are ready. The sections below are purely plots,
statistical tests, and interpretation.

---

## 7. IMD Baseline & Change

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(msoa_wealth['IMD_2010'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Mean IMD 2010 Score (higher = more deprived)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Chart 1A: IMD 2010 Score Distribution (London MSOAs)')

dc = msoa_wealth['Wealth_Decile'].value_counts().sort_index()
axes[1].bar(dc.index, dc.values, color='teal', edgecolor='white')
axes[1].set_xlabel('Wealth Decile (1 = most deprived, 10 = wealthiest)')
axes[1].set_ylabel('Number of MSOAs')
axes[1].set_title('Chart 1B: MSOA Count per Wealth Decile (2010 Baseline)')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig1_imd_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(msoa_wealth['IMD_Change'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('IMD Change (2019 − 2010)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Chart 2A: IMD Score Change across London MSOAs')

decile_change = msoa_wealth.groupby('Wealth_Decile')['IMD_Change'].mean()
colors = ['coral' if v < 0 else 'steelblue' for v in decile_change.values]
axes[1].bar(decile_change.index, decile_change.values, color=colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Wealth Decile (2010 baseline)')
axes[1].set_ylabel('Mean IMD Change')
axes[1].set_title('Chart 2B: Average IMD Change by 2010 Wealth Decile')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig2_imd_change.png', dpi=150, bbox_inches='tight')
plt.show()

n_less = (msoa_wealth['IMD_Change'] < 0).sum()
n_total = len(msoa_wealth)
print(f'MSOAs that became less deprived: {n_less} ({n_less/n_total*100:.1f}%)')

---
## 8. Flow Heatmaps: 2011 vs 2021 Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, fm, year in [(axes[0], flow_matrix_2011, '2011'),
                      (axes[1], flow_matrix_2021, '2021')]:
    sns.heatmap(
        fm, annot=True, fmt=',.0f', cmap='YlOrRd',
        linewidths=0.5, linecolor='white',
        cbar_kws={'label': 'Number of migrants'}, ax=ax
    )
    ax.set_xlabel('Destination Wealth Decile')
    ax.set_ylabel('Origin Wealth Decile')
    ax.set_title(f'Migration Flows between Wealth Deciles\n({year} Census | Fixed IMD 2010 Deciles)')
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig3_od_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

Compare the two heatmaps:
- Has the diagonal dominance (same-decile moves) changed?
- Have off-diagonal flows (cross-decile moves) intensified?
- Look especially at flows from wealthy origins to deprived destinations (upper-left quadrant).

---
## 9. Decile Shift Distribution: 2011 vs 2021

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, sd, year in [(axes[0], shift_dist_2011, '2011'),
                      (axes[1], shift_dist_2021, '2021')]:
    colors = ['#d73027' if x < 0 else '#4575b4' if x > 0 else '#999999' for x in sd.index]
    ax.bar(sd.index, sd.values, color=colors, edgecolor='white')
    ax.set_xlabel('Decile Shift (negative = moved to more deprived area)')
    ax.set_ylabel('Number of migrants')
    ax.set_title(f'Wealth-Decile Shifts ({year})')
    ax.set_xticks(range(sd.index.min(), sd.index.max() + 1))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig4_decile_shift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Cascade Pressure by Decile: 2011 vs 2021

In [ ]:
print('=== Decile Summary (both years) ===')
display(decile_summary)

In [ ]:
# Paired bar chart: average Net Cascade by decile, 2011 vs 2021
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(1, 11)
width = 0.35

ax.bar(x - width/2, decile_summary['Avg_Net_Cascade_11'], width,
       label='2011', color='#4575b4', edgecolor='white')
ax.bar(x + width/2, decile_summary['Avg_Net_Cascade_21'], width,
       label='2021', color='#d73027', edgecolor='white')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Wealth Decile (2010 baseline)')
ax.set_ylabel('Mean Net Cascade per MSOA')
ax.set_title('Average Cascade Pressure by Decile: 2011 vs 2021')
ax.set_xticks(x)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig5_cascade_by_decile.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monotonicity tests for both years
for col, label in [('Avg_Net_Cascade_11', '2011'), ('Avg_Net_Cascade_21', '2021')]:
    r, p = stats.spearmanr(decile_summary.index, decile_summary[col])
    print(f'Wealth Decile vs Net Cascade ({label}): rho = {r:+.3f}, p = {p:.6f}')

# ΔCFI vs IMD change across deciles
r, p = stats.spearmanr(decile_summary['Avg_Delta_CFI'], decile_summary['Avg_IMD_Change'])
print(f'ΔCFI vs IMD Change (decile-level):        rho = {r:+.3f}, p = {p:.6f}')

---
## 11. ΔCFI Analysis: Where Did Cascade Pressure Intensify?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Distribution of ΔCFI
axes[0].hist(msoa_analysis['Delta_Net_Cascade'], bins=40,
             color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('ΔCFI (Net_Cascade₂₀₂₁ − Net_Cascade₂₀₁₁)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Distribution of Cascade Pressure Change')

# ΔCFI by baseline decile
delta_by_decile = msoa_analysis.groupby('Wealth_Decile')['Delta_Net_Cascade'].mean()
colors = ['coral' if v > 0 else 'steelblue' for v in delta_by_decile.values]
axes[1].bar(delta_by_decile.index, delta_by_decile.values, color=colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Wealth Decile (2010 baseline)')
axes[1].set_ylabel('Mean ΔCFI')
axes[1].set_title('Average ΔCFI by 2010 Wealth Decile')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig6_delta_cfi.png', dpi=150, bbox_inches='tight')
plt.show()

n_intensified = (msoa_analysis['Delta_Net_Cascade'] > 0).sum()
n_weakened = (msoa_analysis['Delta_Net_Cascade'] < 0).sum()
print(f'MSOAs where cascade pressure intensified: {n_intensified}')
print(f'MSOAs where cascade pressure weakened:    {n_weakened}')

##### NOTE

- Positive ΔCFI means cascade pressure intensified between 2011 and 2021.
- If deprived deciles (1–3) show the largest positive ΔCFI, this supports the
  hypothesis that gentrification pressure is accelerating in the poorest areas.
- The COVID caveat applies: 2021 Census migration may be atypical.

---
## 12. MSOA Typology

In [ ]:
# Scatter: 2011 vs 2021 Net Cascade, coloured by typology
type_colors = {'Emerging': '#d73027', 'Sustained': '#fc8d59',
               'Stalled': '#91bfdb', 'Stable': '#4575b4'}

fig, ax = plt.subplots(figsize=(9, 8))
for gtype, color in type_colors.items():
    mask = msoa_analysis['Gentrif_Type'] == gtype
    ax.scatter(
        msoa_analysis.loc[mask, 'Net_Cascade_11'],
        msoa_analysis.loc[mask, 'Net_Cascade_21'],
        c=color, label=gtype, s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
    )

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
# Identity line
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'k:', alpha=0.3, label='No change')
ax.set_xlabel('Net Cascade (2011)')
ax.set_ylabel('Net Cascade (2021)')
ax.set_title('MSOA Gentrification Typology\n(2011 vs 2021 Cascade Pressure)')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig7_typology_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Typology summary
type_summary = msoa_analysis.groupby('Gentrif_Type').agg(
    Count=('msoa11cd', 'count'),
    Avg_IMD_2010=('IMD_2010', 'mean'),
    Avg_IMD_Change=('IMD_Change', 'mean'),
    Avg_Delta_CFI=('Delta_Net_Cascade', 'mean'),
).round(2)
display(type_summary)

##### NOTE

- **Emerging**: no cascade pressure in 2011, but present in 2021 — newly gentrifying.
- **Sustained**: cascade pressure in both periods — ongoing gentrification.
- **Stalled**: had cascade pressure in 2011 but lost it by 2021.
- **Stable**: no cascade pressure in either period.

If the index is valid, Emerging + Sustained MSOAs should have higher average
IMD decline than Stable MSOAs.

---
## 13. Validation: ΔCFI vs IMD Change (Three-Way Triangulation)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 13A: ΔCFI vs IMD Change
ax = axes[0]
sc = ax.scatter(
    msoa_analysis['Delta_Net_Cascade'], msoa_analysis['IMD_Change'],
    c=msoa_analysis['Wealth_Decile'], cmap='RdYlGn',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('ΔCFI (2021 − 2011)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('13A: ΔCFI vs Deprivation Change')

# 13B: 2021 Net Cascade vs IMD Change (replicates v2 validation)
ax = axes[1]
ax.scatter(
    msoa_analysis['Net_Cascade_21'], msoa_analysis['IMD_Change'],
    c=msoa_analysis['Wealth_Decile'], cmap='RdYlGn',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Net Cascade (2021)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('13B: 2021 Cascade vs Deprivation Change')

# 13C: 2011 Net Cascade vs IMD Change
ax = axes[2]
ax.scatter(
    msoa_analysis['Net_Cascade_11'], msoa_analysis['IMD_Change'],
    c=msoa_analysis['Wealth_Decile'], cmap='RdYlGn',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Net Cascade (2011)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('13C: 2011 Cascade vs Deprivation Change')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig8_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation statistics
print('=== Validation Correlations (MSOA-level Pearson r) ===')
for col, label in [
    ('Net_Cascade_11', 'Net Cascade 2011'),
    ('Net_Cascade_21', 'Net Cascade 2021'),
    ('Delta_Net_Cascade', 'ΔCFI (2021−2011)'),
    ('Pct_Inflow_Wealthier_21', '% Inflow Wealthier 2021'),
    ('Delta_Pct_Inflow_Wealthier', 'Δ% Inflow Wealthier'),
]:
    valid = msoa_analysis[[col, 'IMD_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Change'])
    print(f'  {label:35s} vs IMD Change:  r = {r:+.3f},  p = {p:.4f}')

##### NOTE

Three-way triangulation:
- If both CFI₂₀₁₁ and CFI₂₀₂₁ correlate with IMD change, the flow-based index is consistent.
- If ΔCFI also correlates with IMD change, it shows that *intensification* of cascade pressure
  predicts larger deprivation decline — a stronger gentrification signal.
- Compare r values: is ΔCFI a stronger predictor than the level at either time point?

---
## 14. Robustness: IMD 2019 Decile Sensitivity Check

In [ ]:
# Do the same MSOAs show up as gentrifying under both baseline choices?
valid = msoa_analysis[['Net_Cascade_21', 'Net_Cascade_21_alt', 'IMD_Change']].dropna()

r_baseline, p_baseline = stats.pearsonr(valid['Net_Cascade_21'], valid['Net_Cascade_21_alt'])
print(f'Net Cascade (IMD 2010 deciles) vs (IMD 2019 deciles): r = {r_baseline:+.3f}, p = {p_baseline:.4f}')

r_alt, p_alt = stats.pearsonr(valid['Net_Cascade_21_alt'], valid['IMD_Change'])
print(f'Net Cascade (IMD 2019 deciles) vs IMD Change:         r = {r_alt:+.3f},   p = {p_alt:.4f}')

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(valid['Net_Cascade_21'], valid['Net_Cascade_21_alt'],
           s=15, alpha=0.5, edgecolors='grey', linewidth=0.3)
lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1])]
ax.plot(lims, lims, 'r--', alpha=0.5, label='Identity line')
ax.set_xlabel('Net Cascade (IMD 2010 deciles)')
ax.set_ylabel('Net Cascade (IMD 2019 deciles)')
ax.set_title(f'Robustness Check: Baseline Sensitivity\nr = {r_baseline:.3f}')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig9_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

If the correlation between the two baseline versions is high (r > 0.8),
results are robust to baseline choice. If it's low, the choice of IMD year
materially affects which MSOAs are identified as gentrifying.

---
## 15. Borough-Level Summary

In [ ]:
display(borough_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))

# Borough cascade: 2021 level
bs = borough_summary.sort_values('Total_Net_Cascade_21', ascending=True)
colors = ['coral' if v > 0 else 'steelblue' for v in bs['Total_Net_Cascade_21']]
axes[0].barh(bs.index, bs['Total_Net_Cascade_21'], color=colors, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('Total Net Cascade (2021)')
axes[0].set_title('Net Cascade by Borough (2021)')

# Borough ΔCFI
bs2 = borough_summary.sort_values('Total_Delta_CFI', ascending=True)
colors2 = ['coral' if v > 0 else 'steelblue' for v in bs2['Total_Delta_CFI']]
axes[1].barh(bs2.index, bs2['Total_Delta_CFI'], color=colors2, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Total ΔCFI (2021 − 2011)')
axes[1].set_title('Change in Cascade Pressure by Borough')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v3_fig10_borough_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 16. Export

In [ ]:
msoa_analysis.to_csv(OUTPUT_DIR / 'v3_msoa_cascade_features.csv', index=False)
borough_summary.to_csv(OUTPUT_DIR / 'v3_borough_summary.csv')
flow_matrix_2011.to_csv(OUTPUT_DIR / 'v3_flow_matrix_2011.csv')
flow_matrix_2021.to_csv(OUTPUT_DIR / 'v3_flow_matrix_2021.csv')
decile_summary.to_csv(OUTPUT_DIR / 'v3_decile_summary.csv')

print('Exported to outputs/:')
print('  - v3_msoa_cascade_features.csv')
print('  - v3_borough_summary.csv')
print('  - v3_flow_matrix_2011.csv')
print('  - v3_flow_matrix_2021.csv')
print('  - v3_decile_summary.csv')

---
## Discussion

1. **Two-period methodology**: Fixed 2010 IMD baseline applied to both 2011 and 2021 O-D
   data. ΔCFI isolates changes in flow dynamics from area reclassification.

2. **Key questions from results**:
   - Does ΔCFI outperform single-period CFI as a gentrification predictor?
   - Which boroughs shifted most between the two periods?
   - Is the typology (Emerging/Sustained/Stalled/Stable) consistent with known gentrification narratives?

3. **COVID caveat**: The 2021 Census captured migration during COVID. Comparing against
   a pre-COVID 2011 baseline makes this explicit rather than hidden.

4. **Robustness**: If IMD 2010 vs 2019 decile baselines produce similar results,
   the index is robust. If not, discuss implications for baseline choice.

5. **2011 O-D data notes**: Aggregated from OA-level WICID headcount data.
   Check whether the OA-to-MSOA aggregation introduces any edge effects.
   Also note whether the count variable is a rounded microdata count.

6. **Next steps**:
   - Spatial mapping (geopandas) with typology colours
   - Alluvial / Sankey plots showing flow changes 2011 → 2021
   - MSOA-level displacement trajectories (top inflow/outflow origins)
   - Flow distance analysis (MSOA centroid distances)
   - NS-SEC decomposition (ODMG04EW) if available
   - Population-at-risk normalisation